# Knowledge Distillation: Training a Smaller Model from a Larger One

**Goal:** use a large "teacher" model to generate labelled data and soft targets, then train a smaller "student" model that approaches teacher-level quality at a fraction of the cost.

---

## The Core Idea

A large model is accurate but expensive to run. A small model is cheap but less accurate.

Knowledge distillation bridges the gap: the teacher shares what it has learned — not just hard labels, but its full probability distribution — and the student learns to mimic that richer signal.

```text
                 ┌────────────────┐
  Unlabelled or  │  Teacher model  │  Large, expensive, accurate
  labelled data  │  (frozen)       │
                 └───────┬────────┘
                         │
              soft labels + hard labels
                         │
                         ▼
                 ┌────────────────┐
                 │  Student model  │  Small, cheap, deployed
                 │  (trainable)    │
                 └────────────────┘
```

## Why Distillation Works

When a teacher classifies an image of a cat, it does not just output "cat." Its softmax distribution might say:

| Class | Probability |
|---|---|
| cat | 0.85 |
| dog | 0.10 |
| fox | 0.04 |
| rabbit | 0.01 |

That distribution contains **dark knowledge**: the teacher thinks cats look a bit like dogs and foxes, but nothing like rabbits. This inter-class similarity information is lost in a hard "cat" label.

The student learns from the full distribution, which is a much richer training signal per example.

## What This Notebook Covers

1. When and why to use distillation
2. Two distillation approaches: logit distillation vs. data distillation
3. Building a teacher-labelled dataset
4. Training a student with a combined distillation + hard-label loss
5. Comparing the student to the teacher and a standalone baseline
6. Temperature, alpha, and practical constraints


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy scikit-learn datasets transformers accelerate torch

In [ ]:
import json
import math
import random
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RUN_TEACHER_LABELLING = False
RUN_TRAINING = False

print(f"Project dir:         {PROJECT_DIR}")
print(f"Artifacts:           {ARTIFACT_DIR}")
print(f"Teacher labelling:   {RUN_TEACHER_LABELLING}")
print(f"Student training:    {RUN_TRAINING}")

## 2. When and Why to Use Distillation

### Good Fits

| Situation | Why distillation helps |
|---|---|
| You need to deploy on edge / mobile / CPU | Smaller student runs within resource limits |
| API cost is dominant | Student replaces expensive teacher API calls |
| Latency matters (real-time serving) | Small model responds faster |
| You have unlabelled data and a strong teacher | Teacher generates labels cheaply for the student |
| Privacy constraints | Distil on-premises rather than sending data to a large cloud model |

### Weak Fits

| Situation | Better approach |
|---|---|
| Task is trivial (linear classifier suffices) | No need for a neural student |
| Teacher itself is bad | Fix the teacher first — garbage in, garbage out |
| You need the student to exceed the teacher | Distillation transfers, it does not amplify |
| Data is very small (< 100 examples) | Fine-tune directly on labelled data |

### Rule of Thumb

Use distillation when you have a **capable teacher** and need a **cheaper student** that retains most of that capability.


## 3. Two Distillation Approaches

### Approach A — Logit (Soft-Label) Distillation

The teacher and student process the **same input**. The student's loss combines:
- cross-entropy against the **hard label** (ground truth)
- KL divergence against the **teacher's soft probabilities** (temperature-scaled softmax)

$$
\mathcal{L} = \alpha \cdot T^2 \cdot \text{KL}\!\left(\sigma(z_s / T),\; \sigma(z_t / T)\right) + (1 - \alpha) \cdot \text{CE}(z_s,\; y)
$$

where $z_s$ and $z_t$ are student and teacher logits, $T$ is temperature, $\alpha$ balances the two losses.

**Requires:** running both the teacher and student during training.

### Approach B — Data Distillation (Teacher-as-Labeller)

The teacher labels a large pool of unlabelled (or weakly-labelled) examples. The student then trains on that teacher-labelled data with standard supervised learning.

**Requires:** teacher inference pass only (offline, one-time).

### Which to Choose?

| Factor | Logit distillation | Data distillation |
|---|---|---|
| **Richer signal** | Yes — full softmax distribution | No — only the argmax label |
| **Requires teacher at training time** | Yes | No (label offline, train later) |
| **Simpler pipeline** | No (custom loss function) | Yes (standard SFT on teacher labels) |
| **Works with API-only teachers** | Only if the API exposes logprobs | Yes — only needs the text output |

This notebook demonstrates **both approaches** so you can compare them.


## 4. The Task: Intent Classification

We use a **5-class intent classification** task for a developer assistant:

| Class | Example |
|---|---|
| `bug_report` | "The login endpoint returns 500 after the last deploy" |
| `feature_request` | "Can you add dark mode to the dashboard?" |
| `how_to` | "How do I set up CI/CD with GitHub Actions?" |
| `account` | "I need to reset my API key" |
| `pricing` | "What's the difference between the free and pro plans?" |

This is a practical distillation scenario: a large LLM classifies intents
accurately but is too expensive for real-time ticket triage. A small classifier
could handle it at 100× lower cost.


In [ ]:
CLASSES = ["bug_report", "feature_request", "how_to", "account", "pricing"]
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}

examples = [
    # ── bug_report ──
    {"text": "The login endpoint returns 500 after the last deploy.", "label": "bug_report"},
    {"text": "Dashboard crashes when I filter by date range.", "label": "bug_report"},
    {"text": "API returns empty response body on POST requests.", "label": "bug_report"},
    {"text": "Error: connection refused when running the CLI tool.", "label": "bug_report"},
    {"text": "Push notifications stopped working after the update.", "label": "bug_report"},
    {"text": "Search results show deleted items intermittently.", "label": "bug_report"},
    {"text": "Memory leak in the worker process after 24 hours.", "label": "bug_report"},
    {"text": "The export CSV feature truncates rows beyond 10k.", "label": "bug_report"},
    {"text": "OAuth callback fails for Google accounts.", "label": "bug_report"},
    {"text": "Webhook delivery keeps timing out on large payloads.", "label": "bug_report"},
    {"text": "The billing page shows last month's invoice twice.", "label": "bug_report"},
    {"text": "Cannot delete a team member from the admin panel.", "label": "bug_report"},

    # ── feature_request ──
    {"text": "Can you add dark mode to the dashboard?", "label": "feature_request"},
    {"text": "It would be great to have Slack integration.", "label": "feature_request"},
    {"text": "Please add support for custom webhook headers.", "label": "feature_request"},
    {"text": "We need an audit log for compliance.", "label": "feature_request"},
    {"text": "Can you add two-factor authentication options?", "label": "feature_request"},
    {"text": "I'd love a mobile app for checking status on the go.", "label": "feature_request"},
    {"text": "Please add batch operations to the REST API.", "label": "feature_request"},
    {"text": "A Terraform provider would be really useful.", "label": "feature_request"},
    {"text": "Can we get granular permissions per project?", "label": "feature_request"},
    {"text": "Would love an option to schedule deployments.", "label": "feature_request"},
    {"text": "Add CSV import for bulk user creation.", "label": "feature_request"},
    {"text": "Can you support SAML SSO for enterprise accounts?", "label": "feature_request"},

    # ── how_to ──
    {"text": "How do I set up CI/CD with GitHub Actions?", "label": "how_to"},
    {"text": "What's the best way to configure environment variables?", "label": "how_to"},
    {"text": "How do I connect my database to the platform?", "label": "how_to"},
    {"text": "Where can I find the API documentation?", "label": "how_to"},
    {"text": "How do I enable logging for debugging?", "label": "how_to"},
    {"text": "What's the recommended way to handle rate limiting?", "label": "how_to"},
    {"text": "How do I migrate from v1 to v2 of the API?", "label": "how_to"},
    {"text": "Can you explain how to set up staging and production environments?", "label": "how_to"},
    {"text": "How do I rotate my API keys without downtime?", "label": "how_to"},
    {"text": "What's the process for setting up a custom domain?", "label": "how_to"},
    {"text": "How do I configure webhook retries?", "label": "how_to"},
    {"text": "How do I set up monitoring alerts for my services?", "label": "how_to"},

    # ── account ──
    {"text": "I need to reset my API key.", "label": "account"},
    {"text": "How do I change the email on my account?", "label": "account"},
    {"text": "Can I transfer ownership of a project to another user?", "label": "account"},
    {"text": "I want to delete my account and all associated data.", "label": "account"},
    {"text": "How do I add a new team member?", "label": "account"},
    {"text": "I'm locked out of my account after too many failed logins.", "label": "account"},
    {"text": "Can I merge two accounts into one?", "label": "account"},
    {"text": "I need to update the billing contact on my account.", "label": "account"},
    {"text": "How do I enable SSO for my organization?", "label": "account"},
    {"text": "Can I downgrade my plan without losing data?", "label": "account"},
    {"text": "I need an invoice for my last three months.", "label": "account"},
    {"text": "My colleague left the company. How do I remove their access?", "label": "account"},

    # ── pricing ──
    {"text": "What's the difference between the free and pro plans?", "label": "pricing"},
    {"text": "How much does the enterprise tier cost?", "label": "pricing"},
    {"text": "Is there a discount for annual billing?", "label": "pricing"},
    {"text": "Do you offer educational or nonprofit pricing?", "label": "pricing"},
    {"text": "What happens if I exceed the API call limit on the free plan?",  "label": "pricing"},
    {"text": "Can I try the pro plan before committing?", "label": "pricing"},
    {"text": "Are there any hidden fees beyond the listed price?", "label": "pricing"},
    {"text": "How does usage-based pricing work?", "label": "pricing"},
    {"text": "Do you charge per seat or per project?", "label": "pricing"},
    {"text": "What payment methods do you accept?", "label": "pricing"},
    {"text": "Can I get a custom quote for our team of 200?", "label": "pricing"},
    {"text": "Is there a startup program with credits?", "label": "pricing"},
]

df = pd.DataFrame(examples)
print(f"Total examples: {len(df)}")
print(f"\nClass distribution:")
print(df["label"].value_counts().to_string())

## 5. Train / Test Split

We stratify by class. The student trains on teacher-labelled data; the test set
uses **ground-truth labels only** so we measure real accuracy, not how well the
student copies the teacher.


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=SEED, stratify=df["label"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}")
print(f"Test:  {len(test_df)}")
print(f"\nTrain class counts: {dict(Counter(train_df['label']))}")
print(f"Test class counts:  {dict(Counter(test_df['label']))}")

## 6. The Teacher Model

### Choosing a Teacher

The teacher must be **meaningfully more capable** than the student. Common setups:

| Teacher | Student | Domain |
|---|---|---|
| GPT-4 / Claude | DistilBERT / TinyBERT | Text classification |
| Llama-3-70B | Llama-3-8B | Text generation |
| ViT-Large | ViT-Tiny | Image classification |
| Whisper-Large | Whisper-Small | Speech recognition |

### In This Notebook

We use `Qwen/Qwen2.5-3B-Instruct` as the teacher and a simple two-layer
feedforward network as the student. This mirrors the real pattern: use a large
LLM to label data, then train a tiny classifier.

### Teacher Labelling

The teacher classifies each training example. We record:
- the **hard prediction** (argmax class)
- the **soft probabilities** (full distribution over classes) — used for logit distillation


In [ ]:
TEACHER_MODEL = "Qwen/Qwen2.5-3B-Instruct"

TEACHER_SYSTEM_PROMPT = (
    "You are a support ticket classifier. "
    "Classify each ticket into exactly one of these categories: "
    "bug_report, feature_request, how_to, account, pricing. "
    "Reply with only the category name. No explanation."
)


def build_teacher_prompt(text):
    return [
        {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]


print(f"Teacher model: {TEACHER_MODEL}")
print(f"\nExample prompt:")
print(json.dumps(build_teacher_prompt("The login endpoint returns 500."), indent=2))

In [ ]:
if RUN_TEACHER_LABELLING:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline

    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL, trust_remote_code=True)
    teacher_model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    teacher_gen = hf_pipeline("text-generation", model=teacher_model, tokenizer=teacher_tokenizer)

    teacher_labels = []
    for _, row in train_df.iterrows():
        prompt = teacher_tokenizer.apply_chat_template(
            build_teacher_prompt(row["text"]),
            tokenize=False,
            add_generation_prompt=True,
        )
        output = teacher_gen(prompt, max_new_tokens=10, do_sample=False)
        raw = output[0]["generated_text"][len(prompt):].strip().lower()
        # Normalise to closest class
        best = min(CLASSES, key=lambda c: 0 if c in raw else 1)
        teacher_labels.append(best)

    train_df["teacher_label"] = teacher_labels
    acc = accuracy_score(train_df["label"], train_df["teacher_label"])
    print(f"Teacher accuracy on train set: {acc:.3f}")
    print(f"Teacher agreement by class:")
    for cls in CLASSES:
        mask = train_df["label"] == cls
        cls_acc = accuracy_score(train_df.loc[mask, "label"], train_df.loc[mask, "teacher_label"])
        print(f"  {cls}: {cls_acc:.3f}")
else:
    # Simulate teacher labels: assume 95% teacher accuracy
    rng = np.random.default_rng(SEED)
    teacher_labels = []
    for _, row in train_df.iterrows():
        if rng.random() < 0.95:
            teacher_labels.append(row["label"])
        else:
            others = [c for c in CLASSES if c != row["label"]]
            teacher_labels.append(rng.choice(others))
    train_df["teacher_label"] = teacher_labels
    acc = accuracy_score(train_df["label"], train_df["teacher_label"])
    print(f"Simulated teacher accuracy: {acc:.3f}")
    print("Set RUN_TEACHER_LABELLING = True to run the real teacher.")

## 7. Generating Soft Targets

### What Soft Targets Are

Instead of just the hard label ("bug_report"), soft targets give the full probability distribution:

```
[0.82, 0.03, 0.05, 0.08, 0.02]
 bug    feat   how   acct  price
```

This tells the student:
- "This is probably a bug report"
- "But there's an 8% chance it's an account issue"  ← **dark knowledge**

### Temperature Scaling

At temperature $T = 1$, the softmax is sharp (peaks on one class).
At higher $T$, the distribution is smoother, revealing more inter-class relationships.

$$
\text{softmax}(z_i / T) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

The original Hinton et al. (2015) paper recommends $T$ between 2 and 20 depending on the task.


In [ ]:
def softmax_with_temperature(logits, temperature=1.0):
    scaled = logits / temperature
    exp = np.exp(scaled - np.max(scaled))  # numerical stability
    return exp / exp.sum()


# Demonstrate temperature effect on a mock logit vector
mock_logits = np.array([4.2, 1.1, 0.8, 2.0, 0.3])

temperatures = [0.5, 1.0, 2.0, 5.0, 10.0]
temp_df = pd.DataFrame(index=CLASSES)
for t in temperatures:
    temp_df[f"T={t}"] = softmax_with_temperature(mock_logits, temperature=t).round(4)

print("Effect of temperature on softmax distribution:")
print(temp_df.to_string())
print("\nHigher T → smoother distribution → more 'dark knowledge' shared with the student.")

In [ ]:
# Generate synthetic soft targets for the training set
# (In practice, these come from the teacher model's output logits)

def generate_soft_targets(labels, n_classes, teacher_confidence=0.85, temperature=3.0):
    rng2 = np.random.default_rng(SEED + 7)
    targets = []
    for label in labels:
        lid = label2id[label]
        # Generate mock logits: high for the teacher's predicted class, lower for others
        logits = rng2.normal(0, 1, n_classes)
        logits[lid] += 4.0  # boost the predicted class
        soft = softmax_with_temperature(logits, temperature=temperature)
        targets.append(soft.tolist())
    return targets


train_df["soft_targets"] = generate_soft_targets(
    train_df["teacher_label"].tolist(),
    len(CLASSES),
    temperature=3.0,
)

# Show a few examples
for i in range(3):
    row = train_df.iloc[i]
    print(f"Text:   {row['text'][:60]}...")
    print(f"Label:  {row['label']}")
    print(f"Teacher: {row['teacher_label']}")
    probs = row['soft_targets']
    print(f"Soft:   {' | '.join(f'{c}: {p:.3f}' for c, p in zip(CLASSES, probs))}")
    print()

## 8. Student Model Architecture

### Choosing Student Size

The student must be small enough to justify distillation. If it is almost as large
as the teacher, you might as well use the teacher.

| Teacher | Student | Compression ratio |
|---|---|---|
| 3B param LLM | 2-layer MLP (50K params) | ~60,000× smaller |
| BERT-Large (340M) | DistilBERT (66M) | ~5× smaller |
| ViT-Large (307M) | ViT-Tiny (5.7M) | ~54× smaller |

### Our Student

We use a simple two-layer feedforward classifier on top of a pre-computed
sentence embedding. This is intentionally minimal to show the distillation
concept clearly.


In [ ]:
from transformers import AutoTokenizer, AutoModel

STUDENT_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ENCODER)
student_encoder = AutoModel.from_pretrained(STUDENT_ENCODER)
student_encoder.eval()

EMBED_DIM = student_encoder.config.hidden_size

print(f"Student encoder: {STUDENT_ENCODER}")
print(f"Embedding dim:   {EMBED_DIM}")
print(f"Encoder params:  {sum(p.numel() for p in student_encoder.parameters()):,}")

In [ ]:
def encode_texts(texts, tokenizer, encoder, batch_size=32):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
        with torch.no_grad():
            outputs = encoder(**inputs)
        # Mean pooling
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeds.append(embeddings)
    return torch.cat(all_embeds, dim=0)


train_embeddings = encode_texts(train_df["text"].tolist(), student_tokenizer, student_encoder)
test_embeddings = encode_texts(test_df["text"].tolist(), student_tokenizer, student_encoder)

print(f"Train embeddings: {train_embeddings.shape}")
print(f"Test embeddings:  {test_embeddings.shape}")

In [ ]:
class StudentClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


student_model = StudentClassifier(EMBED_DIM, 256, len(CLASSES))
student_params = sum(p.numel() for p in student_model.parameters())
print(f"Student classifier params: {student_params:,}")
print(f"Architecture:\n{student_model}")

## 9. The Distillation Loss

### Combined Loss

The student optimises a weighted combination of two objectives:

1. **Hard loss** — standard cross-entropy against the ground-truth label
2. **Soft loss** — KL divergence between the student's temperature-scaled softmax
   and the teacher's temperature-scaled soft targets

$$
\mathcal{L}_{\text{distill}} = \alpha \cdot T^2 \cdot \text{KL}(\sigma(z_s/T) \| \sigma(z_t/T)) + (1 - \alpha) \cdot \text{CE}(z_s, y)
$$

### Why Multiply by $T^2$?

The gradients of the KL term are scaled down by $1/T^2$ when temperature is applied.
Multiplying by $T^2$ compensates, keeping the gradient magnitudes balanced with the
hard loss regardless of the temperature value.

### Choosing Alpha ($\alpha$)

| $\alpha$ | Behaviour | When to use |
|---|---|---|
| 0.0 | Pure hard label training | No distillation (baseline) |
| 0.3 – 0.5 | Balanced | Good default for most tasks |
| 0.7 – 0.9 | Mostly teacher-guided | Very confident teacher, or no hard labels |
| 1.0 | Pure soft label | Labels are entirely teacher-generated |


In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, temperature=3.0, alpha=0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_loss = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_soft_targets, hard_labels):
        # Hard loss: standard cross-entropy
        hard_loss = self.ce_loss(student_logits, hard_labels)

        # Soft loss: KL divergence at temperature T
        T = self.temperature
        student_soft = F.log_softmax(student_logits / T, dim=-1)
        teacher_soft = torch.tensor(teacher_soft_targets, dtype=torch.float32)
        # Re-normalise teacher targets at temperature T (they were already generated at T)
        soft_loss = self.kl_loss(student_soft, teacher_soft)

        # Combined
        loss = self.alpha * T * T * soft_loss + (1 - self.alpha) * hard_loss
        return loss, hard_loss.item(), soft_loss.item()


distill_loss_fn = DistillationLoss(temperature=3.0, alpha=0.5)
print(f"Temperature: {distill_loss_fn.temperature}")
print(f"Alpha:       {distill_loss_fn.alpha}")
print("Loss components: hard (CE) + soft (KL)")

## 10. Training the Student

We train three variants for comparison:

| Variant | Loss | What it shows |
|---|---|---|
| **Baseline** | CE on hard labels only | How well the student learns without the teacher |
| **Data-distilled** | CE on teacher's hard labels | Student trained on teacher-generated labels |
| **Logit-distilled** | CE + KL on soft targets | Full distillation with dark knowledge |


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 40
LR = 1e-3
BATCH_SIZE = 16

# Prepare labels
hard_labels = torch.tensor([label2id[l] for l in train_df["label"]], dtype=torch.long)
teacher_hard_labels = torch.tensor([label2id[l] for l in train_df["teacher_label"]], dtype=torch.long)
soft_targets = train_df["soft_targets"].tolist()

test_labels = torch.tensor([label2id[l] for l in test_df["label"]], dtype=torch.long)

# DataLoaders
train_dataset = TensorDataset(train_embeddings, hard_labels, teacher_hard_labels)
test_dataset = TensorDataset(test_embeddings, test_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader):>3}   Epochs: {EPOCHS}   LR: {LR}")

In [ ]:
def train_student(mode="baseline"):
    model = StudentClassifier(EMBED_DIM, 256, len(CLASSES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    ce_fn = nn.CrossEntropyLoss()
    distill_fn = DistillationLoss(temperature=3.0, alpha=0.5)
    history = {"train_loss": [], "test_acc": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            embeds, labels_hard, labels_teacher = batch
            logits = model(embeds)

            if mode == "baseline":
                loss = ce_fn(logits, labels_hard)
            elif mode == "data_distilled":
                loss = ce_fn(logits, labels_teacher)
            elif mode == "logit_distilled":
                # Get soft targets for this batch
                indices = []
                for i, (e, lh, lt) in enumerate(zip(embeds, labels_hard, labels_teacher)):
                    # Match by embedding to get index in train_df
                    # (in practice, store the index in the dataset)
                    pass
                # Simplified: we use all soft targets indexed in order
                batch_start = (epoch_loss == 0.0)  # heuristic — use direct index below
                loss, _, _ = distill_fn(logits, soft_targets[:len(logits)], labels_hard)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(labels_hard)

        epoch_loss /= len(train_loader.dataset)
        history["train_loss"].append(epoch_loss)

        # Test accuracy
        model.eval()
        all_preds = []
        with torch.no_grad():
            for batch in test_loader:
                embeds_t, labels_t = batch
                logits_t = model(embeds_t)
                preds = logits_t.argmax(dim=-1).tolist()
                all_preds.extend(preds)
        test_acc = accuracy_score(test_df["label"].map(label2id).tolist(), all_preds)
        history["test_acc"].append(test_acc)

    return model, history


# We use a cleaner indexed approach for the logit-distilled variant
def train_logit_distilled():
    model = StudentClassifier(EMBED_DIM, 256, len(CLASSES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    distill_fn = DistillationLoss(temperature=3.0, alpha=0.5)
    history = {"train_loss": [], "test_acc": []}

    # Index-aware dataset
    indices = torch.arange(len(train_embeddings))
    indexed_ds = TensorDataset(train_embeddings, hard_labels, indices)
    indexed_loader = DataLoader(indexed_ds, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for embeds, labels_hard, idxs in indexed_loader:
            logits = model(embeds)
            batch_soft = [soft_targets[i] for i in idxs.tolist()]
            loss, _, _ = distill_fn(logits, batch_soft, labels_hard)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(labels_hard)

        epoch_loss /= len(indexed_loader.dataset)
        history["train_loss"].append(epoch_loss)

        model.eval()
        all_preds = []
        with torch.no_grad():
            for batch in test_loader:
                embeds_t, labels_t = batch
                preds = model(embeds_t).argmax(dim=-1).tolist()
                all_preds.extend(preds)
        test_acc = accuracy_score(test_df["label"].map(label2id).tolist(), all_preds)
        history["test_acc"].append(test_acc)

    return model, history


print("Training functions ready. Three variants: baseline, data_distilled, logit_distilled.")

In [ ]:
if RUN_TRAINING:
    print("Training baseline (hard labels only)...")
    baseline_model, baseline_hist = train_student("baseline")

    print("Training data-distilled (teacher hard labels)...")
    data_model, data_hist = train_student("data_distilled")

    print("Training logit-distilled (soft targets)...")
    logit_model, logit_hist = train_logit_distilled()

    print("\nDone. All three variants trained.")
else:
    # Simulate plausible learning curves for demonstration
    rng3 = np.random.default_rng(SEED + 10)
    epochs_range = list(range(1, EPOCHS + 1))

    def sim_curve(final_acc, speed=0.12, noise=0.02):
        curve = [final_acc * (1 - math.exp(-speed * e)) + rng3.normal(0, noise) for e in epochs_range]
        return [max(0.1, min(1.0, v)) for v in curve]

    baseline_hist = {"train_loss": sim_curve(0.4, 0.15), "test_acc": sim_curve(0.74, 0.10)}
    data_hist = {"train_loss": sim_curve(0.35, 0.15), "test_acc": sim_curve(0.80, 0.11)}
    logit_hist = {"train_loss": sim_curve(0.30, 0.15), "test_acc": sim_curve(0.84, 0.12)}

    print("Using simulated learning curves for demonstration.")
    print("Set RUN_TRAINING = True to run actual training.")

## 11. Compare Student Variants

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
epochs_range = range(1, EPOCHS + 1)

for hist, label, color in [
    (baseline_hist, "Baseline (hard labels)", "#d62728"),
    (data_hist, "Data-distilled (teacher labels)", "#ff7f0e"),
    (logit_hist, "Logit-distilled (soft targets)", "#2ca02c"),
]:
    ax1.plot(epochs_range, hist["train_loss"], label=label, color=color)
    ax2.plot(epochs_range, hist["test_acc"], label=label, color=color)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")
ax1.legend(fontsize=8)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Test Accuracy")
ax2.set_title("Test Accuracy")
ax2.set_ylim([0.3, 1.0])
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nFinal test accuracy:")
for name, hist in [("Baseline", baseline_hist), ("Data-distilled", data_hist), ("Logit-distilled", logit_hist)]:
    print(f"  {name:<22} {hist['test_acc'][-1]:.3f}")

## 12. Temperature and Alpha Sensitivity

### Temperature ($T$)

| $T$ | Effect |
|---|---|
| 1 | Hard softmax — little dark knowledge |
| 2–5 | Good default range — smooths the distribution, reveals inter-class similarities |
| 10–20 | Very smooth — can dilute the signal too much |

### Alpha ($\alpha$)

| $\alpha$ | Effect |
|---|---|
| 0.0 | Ignores teacher entirely (pure supervised) |
| 0.3 | Mostly hard labels, gentle teacher guidance |
| 0.5 | Equal weight — good default |
| 0.7 | Relies heavily on teacher |
| 1.0 | Pure distillation, ignores ground truth |

The best values depend on:
- teacher quality (high → lean more on teacher, higher $\alpha$)
- dataset size (small → rely more on teacher's signal)
- label noise (noisy labels → teacher's signal may be cleaner)


In [ ]:
# Simulate temperature and alpha sweep
sweep_rng = np.random.default_rng(SEED + 20)

temps = [1.0, 2.0, 3.0, 5.0, 10.0]
alphas = [0.0, 0.3, 0.5, 0.7, 1.0]

sweep_results = []
for t in temps:
    for a in alphas:
        # Model final accuracy: peaks around T=3, alpha=0.5 for this simulated setup
        base = 0.75
        t_bonus = -0.01 * (t - 3.0) ** 2 + 0.04
        a_bonus = -0.1 * (a - 0.5) ** 2 + 0.06
        acc = base + t_bonus + a_bonus + sweep_rng.normal(0, 0.01)
        acc = max(0.60, min(0.92, acc))
        sweep_results.append({"temperature": t, "alpha": a, "test_acc": round(acc, 3)})

sweep_df = pd.DataFrame(sweep_results)
pivot = sweep_df.pivot(index="temperature", columns="alpha", values="test_acc")

fig, ax = plt.subplots(figsize=(8, 4))
import seaborn as sns
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn", ax=ax)
ax.set_title("Simulated Test Accuracy by Temperature and Alpha")
ax.set_ylabel("Temperature (T)")
ax.set_xlabel("Alpha (α)")
plt.tight_layout()
plt.show()

print("\nBest configuration:")
best = sweep_df.loc[sweep_df["test_acc"].idxmax()]
print(f"  T={best['temperature']}, α={best['alpha']} → acc={best['test_acc']:.3f}")

## 13. Cost and Latency Analysis

### Why Distillation Saves Money

The teacher runs **once** (offline labelling). The student runs **continuously** in production.

| Model | Params | Inference cost per query | Latency |
|---|---|---|---|
| Teacher (3B LLM) | ~3,000M | High (GPU required) | ~200 ms |
| Student (MLP classifier) | ~50K | Negligible (CPU) | ~2 ms |

Over millions of queries, the cost difference is enormous:


In [ ]:
queries_per_day = 100_000
teacher_cost_per_query = 0.002  # USD (API cost estimate)
student_cost_per_query = 0.00001  # USD (tiny CPU inference)

months = 12
days = months * 30

teacher_total = queries_per_day * days * teacher_cost_per_query
student_total = queries_per_day * days * student_cost_per_query
labelling_cost = len(train_df) * teacher_cost_per_query  # one-time teacher labelling

cost_df = pd.DataFrame({
    "Item": [
        "Teacher (continuous serving)",
        "Student (continuous serving)",
        "Teacher labelling (one-time)",
        "Savings (teacher → student)",
    ],
    "Cost (USD)": [
        f"${teacher_total:,.0f}",
        f"${student_total:,.0f}",
        f"${labelling_cost:,.2f}",
        f"${teacher_total - student_total - labelling_cost:,.0f}",
    ],
    "Period": [f"{months} months", f"{months} months", "one-time", f"{months} months"],
})
print("COST COMPARISON")
print("=" * 60)
print(cost_df.to_string(index=False))
print(f"\nCompression ratio: {3_000_000_000 / 50_000:,.0f}×")
print(f"Latency reduction: ~100×")

## 14. Limitations and Failure Modes

### What Can Go Wrong

| Failure mode | Description | Mitigation |
|---|---|---|
| **Teacher is wrong** | Student copies teacher errors | Always evaluate student on ground-truth labels, not teacher labels |
| **Distribution mismatch** | Teacher labelled data from one domain, student sees another | Label data from the deployment domain |
| **Capacity gap too large** | Student is too small to represent the task | Increase student capacity or use a bigger architecture |
| **Soft targets too uniform** | Temperature is too high, everything looks the same | Reduce temperature |
| **Label leakage** | Teacher was trained on test-like data | Ensure teacher was not trained on the evaluation data |
| **Overfitting the teacher signal** | Student memorises teacher quirks | Regularisation, early stopping, data augmentation |

### The Ceiling

The student can approach but rarely exceed the teacher's accuracy. If you need better
performance than the teacher, you need a better teacher — not more distillation.

### When to Re-distil

- When your data distribution changes (new intents, new domains)
- When you upgrade the teacher model
- When evaluation shows the student drifting below acceptable accuracy


## 15. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "knowledge_distillation",
    "teacher_model": TEACHER_MODEL,
    "student_encoder": STUDENT_ENCODER,
    "student_params": student_params,
    "classes": CLASSES,
    "dataset_size": len(df),
    "train_size": len(train_df),
    "test_size": len(test_df),
    "simulated_final_acc": {
        "baseline": round(baseline_hist["test_acc"][-1], 3),
        "data_distilled": round(data_hist["test_acc"][-1], 3),
        "logit_distilled": round(logit_hist["test_acc"][-1], 3),
    },
}

log_path = ARTIFACT_DIR / "distillation_experiment_log.json"
log_path.write_text(json.dumps(log, indent=2), encoding="utf-8")
print(f"Saved: {log_path}")

## 16. Key Takeaways

### What We Built

- A **teacher-labelling pipeline** that classifies training data using a large LLM
- A **student classifier** trained with three strategies: baseline, data-distilled, and logit-distilled
- A **distillation loss** combining hard cross-entropy and soft KL divergence
- A **temperature and alpha sweep** showing how these hyperparameters affect accuracy
- A **cost analysis** showing the economic case for distillation

### Design Principles

1. **The teacher runs offline; the student runs in production.** This is the economic core of distillation.

2. **Soft targets are more informative than hard labels.** The teacher's full probability distribution shares inter-class similarity information that hard labels do not contain.

3. **Temperature controls how much dark knowledge is shared.** Too low ($T=1$) keeps the distribution sharp; too high ($T>10$) makes it uniformly flat. $T$ between 2 and 5 is a practical default.

4. **Always evaluate on ground-truth labels, not teacher labels.** The student must be accurate on real data, not just good at copying the teacher.

5. **The student's ceiling is the teacher's accuracy.** Distillation transfers knowledge; it does not create it.

6. **Re-distil when the domain shifts or the teacher improves.** A distilled model is a snapshot of the teacher's knowledge at labelling time.

### When to Use Distillation

- You have a strong teacher and need a cheaper deployment model
- You have unlabelled data and the teacher can label it
- Latency, cost, or privacy constraints prevent deploying the teacher directly
- You need a simple, fast classifier for a narrow task that a large model handles well
